# 04 — Shooting Method & Newmark Integration

**Topic**: Shooting method, Newmark average constant acceleration, monodromy matrix.

**Reference**: Krack & Gross (2019) §3.2; Newmark (1959)

**Estimated runtime**: < 20 seconds

## Theory

**Shooting method** (K&G §3): find initial conditions $y_0 = [q_0, \dot{q}_0]$ such that the integrated trajectory is $T$-periodic:

$$R(y_0, \Omega) = \phi(y_0, T) - y_0 = 0, \quad T = 2\pi/\Omega \tag{1}$$

where $\phi(y_0, T)$ is the time-$T$ flow map obtained by integrating the ODE.

**Newmark average constant acceleration** (K&G §3.2, $\beta = 1/4$, $\gamma = 1/2$) is the time integrator:

$$M\ddot{q}_{n+1} + D\dot{q}_{n+1} + Kq_{n+1} + f_{nl}(q_{n+1}, \dot{q}_{n+1}) = f_{ext,n+1} \tag{2}$$

$$q_{n+1} = q_n + h\dot{q}_n + h^2\left[(\tfrac{1}{2}-\beta)\ddot{q}_n + \beta\ddot{q}_{n+1}\right] \tag{3}$$

$$\dot{q}_{n+1} = \dot{q}_n + h\left[(1-\gamma)\ddot{q}_n + \gamma\ddot{q}_{n+1}\right] \tag{4}$$

With $\beta=1/4, \gamma=1/2$ this scheme is **unconditionally stable** and second-order accurate.

**Monodromy matrix** $\Phi = \partial\phi/\partial y_0$ (sensitivity of the flow map to initial conditions) is propagated alongside the trajectory. Its eigenvalues (Floquet multipliers) indicate stability.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import scipy.integrate as integrate
import matplotlib.pyplot as plt

from nlvib.systems.oscillators import SingleMassOscillator
from nlvib.nonlinearities.elements import cubic_spring
from nlvib.solvers.shooting import newmark_step, shooting_residual

print("Imports OK")

In [ ]:
# Compare Newmark vs scipy.integrate.solve_ivp for 1-DOF linear system
# Linear system: m*ddot_q + d*dot_q + k*q = F*cos(omega*t)

m, d, k = 1.0, 0.02, 1.0
F_amp = 0.1
omega = 0.8
T = 2*np.pi / omega

n_steps = 64     # ← try changing this (Newmark accuracy vs step count)
dt = T / n_steps

sys_lin = SingleMassOscillator(m=m, d=d, k=k)  # no nonlinear element

# Newmark integration
y = np.array([0.05, 0.0])  # initial [q, dq]
t_newmark = [0.0]
q_newmark = [y[0]]

def nl_fn_zero(q, dq):
    return np.zeros_like(q)

ddq_prev = None
for i in range(n_steps):
    t_n = i * dt
    f_ext = np.array([F_amp * np.cos(omega * (t_n + dt))])
    y_new, ddq_new = newmark_step(y, f_ext, sys_lin.M, sys_lin.D, sys_lin.K,
                                   nl_fn_zero, dt, beta=0.25, gamma=0.5,
                                   ddq_n=ddq_prev)
    y = y_new
    ddq_prev = ddq_new
    t_newmark.append((i+1) * dt)
    q_newmark.append(y[0])

# scipy solve_ivp reference
def ode(t, y_):
    q_, dq_ = y_
    ddq_ = (F_amp*np.cos(omega*t) - d*dq_ - k*q_) / m
    return [dq_, ddq_]

t_span = (0.0, T)
sol = integrate.solve_ivp(ode, t_span, [0.05, 0.0], method='RK45',
                           t_eval=np.array(t_newmark), rtol=1e-10, atol=1e-12)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_newmark, q_newmark, 'o-', ms=4, label=f'Newmark (n_steps={n_steps})', color='tab:blue')
ax.plot(sol.t, sol.y[0], '--', lw=2, label='scipy RK45 (reference)', color='tab:orange')
ax.set_xlabel('Time $t$ (s)')
ax.set_ylabel('Displacement $q$')
ax.set_title(f'Newmark vs RK45 — 1-DOF linear ($\\Omega={omega}$ rad/s, n_steps={n_steps})')
ax.legend()
ax.grid(True, alpha=0.3)

max_err = np.max(np.abs(np.array(q_newmark) - sol.y[0]))
print(f"Max |Newmark - RK45| = {max_err:.4e}")
fig

In [ ]:
# Show period-1 orbit for the Duffing oscillator via shooting_residual
# Find periodic orbit using a few Newton steps on the shooting residual

m, d, k, k3 = 1.0, 0.02, 1.0, 0.5
F_amp = 0.1
omega_shoot = 1.02  # near resonance
T_shoot = 2*np.pi / omega_shoot
n_steps_shoot = 64  # ← try changing this

sys_duff = SingleMassOscillator(m=m, d=d, k=k)
sys_duff.add_nonlinear_element(cubic_spring(k3=k3, dof_index=0))

def f_ext_fn(t):
    return np.array([F_amp * np.cos(omega_shoot * t)])

# Initial guess: linear amplitude
lin_amp = F_amp / abs(-(omega_shoot**2)*m + k + 1j*omega_shoot*d)
y0 = np.array([float(lin_amp), 0.0])

# Newton solve for periodic orbit
for _ in range(25):
    R, J = shooting_residual(y0, omega_shoot, sys_duff,
                              n_periods=1, n_steps=n_steps_shoot,
                              f_ext_fn=f_ext_fn)
    if np.linalg.norm(R) < 1e-10:
        break
    try:
        y0 = y0 + np.linalg.solve(J, -R)
    except np.linalg.LinAlgError:
        break

print(f"Periodic orbit: y0 = {y0}, ||R|| = {np.linalg.norm(R):.4e}")

# Reconstruct 2 periods for plotting
t_orbit = np.linspace(0, 2*T_shoot, 2*n_steps_shoot+1)
y_track = y0.copy()
q_orbit = [y_track[0]]
dq_orbit = [y_track[1]]
dt_orb = T_shoot / n_steps_shoot
ddq_p = None
for i in range(2*n_steps_shoot):
    t_n = i * dt_orb
    f_e = f_ext_fn(t_n + dt_orb)
    nl_fn_duff = lambda q, dq: np.array([k3 * q[0]**3])
    y_n, ddq_p = newmark_step(y_track, f_e, sys_duff.M, sys_duff.D, sys_duff.K,
                               nl_fn_duff, dt_orb, beta=0.25, gamma=0.5, ddq_n=ddq_p)
    y_track = y_n
    q_orbit.append(y_track[0])
    dq_orbit.append(y_track[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t_orbit, q_orbit, lw=2, color='tab:blue')
ax1.axvline(T_shoot, color='k', ls='--', lw=1, label=f'$T={T_shoot:.3f}$ s')
ax1.axvline(2*T_shoot, color='k', ls='--', lw=1)
ax1.set_xlabel('Time $t$ (s)')
ax1.set_ylabel('Displacement $q$')
ax1.set_title(f'Period-1 orbit — Duffing, $\\Omega={omega_shoot}$ rad/s')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(q_orbit, dq_orbit, lw=2, color='tab:orange')
ax2.plot(q_orbit[0], dq_orbit[0], 'go', ms=10, label='start ($y_0$)')
ax2.set_xlabel('$q$')
ax2.set_ylabel('$\\dot{q}$')
ax2.set_title('Phase portrait')
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig

## Key Takeaways

- `newmark_step` implements eq. (3)-(4) exactly — unconditionally stable for $\beta=1/4$, $\gamma=1/2$.
- Newmark matches RK45 well for $n_{steps} \geq 32$; error scales as $O(\Delta t^2)$ for this implicit scheme.
- `shooting_residual` returns $R = \phi(y_0, T) - y_0$ and its Jacobian $J = \partial\phi/\partial y_0 - I$ (monodromy-based).
- The phase portrait of the converged orbit is a closed curve — confirming periodicity.